* Kathy Chu
* March 5, 2026
* DSC540 Data Preparation

### Term Project: Milestone 5 

### Project Overview
This project analyzes LendingClub personal loan data to explore the relationship between borrower risk, credit ratings, and macroeconomic conditions at the time of loan issuance. 
The goal is to build a clean, unified dataset from three separate sources that can be used to understand what factors are associated with higher-risk loans.

---

### Data Sources

| Source | Type | Description |
|--------|------|-------------|
| LendingClub | CSV (Flat File) | 2.2M personal loan records with borrower financials, loan terms, and repayment status from 2007–2018 |
| NYU Stern (Damodaran) | Website (HTML) | Credit rating default spreads mapping letter grades to risk percentages |
| FRED (Federal Reserve) | API (JSON) | Monthly U.S. unemployment rate from 2007–2018 |
---

### Project Structure

| Milestone | Description |
|-----------|-------------|
| Milestone 2 | Cleaning the flat file (LendingClub CSV) |
| Milestone 3 | Cleaning website data (NYU Stern credit ratings) |
| Milestone 4 | Connecting to an API and cleaning the data (FRED unemployment) |
| Milestone 5 | Merging all sources into a SQLite database and visualizing in Tableau |

In [33]:
# =============================================================================
# Imports Dependencies

import pandas as pd
import numpy as np
import requests
import sqlite3
import re
from io import StringIO

# =============================================================================
# Data Download
# The LendingClub loan dataset is sourced from Kaggle. Run the lines below
# once to download it locally before running the rest of the notebook.
# Requires a Kaggle account and kagglehub installed: pip install kagglehub
# Kaggle dataset: https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv
# =============================================================================

import kagglehub
path = kagglehub.dataset_download("adarshsng/lending-club-loan-data-csv")
print("Dataset downloaded to:", path)

Dataset downloaded to: /Users/katchu/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv/versions/1


---
## Milestone 2: Cleaning the Flat File

**Data Source:** LendingClub Loan Data (CSV)  
**Retrieved From:** [Kaggle — LendingClub Loan Data](https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv)  
**Description:** Personal loan records issued by LendingClub between 2007–2018, containing
borrower financials, loan terms, and repayment status across 2260668 rows and 145 columns.  
**Note:** `lending_data.csv` is not stored in this repository due to file size. Download it
from the Kaggle link above and place it in the same directory as this notebook before running.

In [34]:
# =============================================================================
# Milestone 2 — Step 0: Load Data and Initial Health Check
# Verifying the dataset dimensions and identifying problem areas before
# any transformations are applied.
# =============================================================================

df = pd.read_csv('/Users/katchu/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv/versions/1/loan.csv', low_memory=False)

print(f"Total Records  : {df.shape[0]}")
print(f"Total Columns  : {df.shape[1]}")

empty_cols = df.columns[df.isnull().all()].tolist()
print(f"\nCompletely Empty Columns : {len(empty_cols)}")

print("\nKey Column Data Types:")
print(df[['grade', 'issue_d', 'int_rate']].dtypes)

missing_stats = df.isnull().sum().sort_values(ascending=False)
print("\nTop 10 Columns with Missing Data:")
print(missing_stats.head(10))

Total Records  : 2260668
Total Columns  : 145

Completely Empty Columns : 3

Key Column Data Types:
grade        object
issue_d      object
int_rate    float64
dtype: object

Top 10 Columns with Missing Data:
id                                            2260668
url                                           2260668
member_id                                     2260668
orig_projected_additional_accrued_interest    2252242
hardship_length                               2250055
hardship_reason                               2250055
hardship_status                               2250055
deferral_term                                 2250055
hardship_amount                               2250055
hardship_start_date                           2250055
dtype: int64


In [35]:
# =============================================================================
# Milestone 2 — Transformation Steps
# =============================================================================

# Step #1: Drop Completely Empty Columns
# The health check revealed columns with zero data across all rows.
# Removing them reduces noise and improves processing performance.
df = df.dropna(axis=1, how='all')

# Step #2: Filter to 2007-2018 Date Range
# Restricting to the advertised dataset range to ensure data completeness
# and to align with the FRED unemployment data we are joining against.
# The date format in this dataset is 'Dec-2018' requiring format='%b-%Y'.
df['issue_d_parsed'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df = df[
    (df['issue_d_parsed'].dt.year >= 2007) &
    (df['issue_d_parsed'].dt.year <= 2018)
].copy()
print(f"Records after date filter : {len(df):,}")

# Step #3: Remove Duplicate Records
# Dropping fully duplicate rows since id, member_id are empty in this version.
df = df.drop_duplicates()

# Step #4: Standardize Column Headers
# Stripping any leading or trailing whitespace from all column names
# to prevent silent key errors in future joins or lookups.
df.columns = [col.strip() for col in df.columns]

# Step #5: Address Missing Values in Employment Title
# Records missing 'emp_title' are labeled 'Unknown' to preserve rows
# rather than dropping valid loan data.
df['emp_title'] = df['emp_title'].fillna('Unknown')

# Step #6: Remove Income Outliers
# The top 1% of annual_inc values represent extreme earners that would
# skew any income-based analysis. Removing them keeps the dataset
# representative of typical borrower behavior.
upper_limit = df['annual_inc'].quantile(0.99)
df_loans = df[df['annual_inc'] <= upper_limit].reset_index(drop=True)

print(f"Final Dataset Shape : {df_loans.shape}")
print(f"\nDate Range : {df_loans['issue_d_parsed'].min().strftime('%b %Y')} to {df_loans['issue_d_parsed'].max().strftime('%b %Y')}")
print("\nFirst 5 Rows of Cleaned Loan Data:")
df_loans.head()

Records after date filter : 2,260,668
Final Dataset Shape : (2238121, 143)

Date Range : Jun 2007 to Dec 2018

First 5 Rows of Cleaned Loan Data:


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,issue_d_parsed
0,2500,2500,2500.0,36 months,13.56,84.92,C,C1,Chef,10+ years,RENT,55000.0,Not Verified,Dec-2018,Current,n,NaN,debt_consolidation,Debt consolidation,109xx,NY,18.24,0.0,Apr-2001,1.0,NaN,45.0,9.0,1.0,4341,10.3,34.0,w,2386.02,2386.02,167.02,167.02,113.98,53.04,0.0,0.0,0.0,Feb-2019,84.92,Mar-2019,Feb-2019,0.0,NaN,1,Individual,NaN,NaN,NaN,0.0,0.0,16901.0,2.0,2.0,1.0,2.0,2.0,12560.0,69.0,2.0,7.0,2137.0,28.0,42000.0,1.0,11.0,2.0,9.0,1878.0,34360.0,5.9,0.0,0.0,140.0,212.0,1.0,1.0,0.0,1.0,NaN,2.0,NaN,0.0,2.0,5.0,3.0,3.0,16.0,7.0,18.0,5.0,9.0,0.0,0.0,0.0,3.0,100.0,0.0,1.0,0.0,60124.0,16901.0,36500.0,18124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2018-12-01
1,30000,30000,30000.0,60 months,18.94,777.23,D,D2,Postmaster,10+ years,MORTGAGE,90000.0,Source Verified,Dec-2018,Current,n,NaN,debt_consolidation,Debt consolidation,713xx,LA,26.52,0.0,Jun-1987,0.0,71.0,75.0,13.0,1.0,12315,24.2,44.0,w,29387.75,29387.75,1507.11,1507.11,612.25,894.86,0.0,0.0,0.0,Feb-2019,777.23,Mar-2019,Feb-2019,0.0,NaN,1,Individual,NaN,NaN,NaN,0.0,1208.0,321915.0,4.0,4.0,2.0,3.0,3.0,87153.0,88.0,4.0,5.0,998.0,57.0,50800.0,2.0,15.0,2.0,10.0,24763.0,13761.0,8.3,0.0,0.0,163.0,378.0,4.0,3.0,3.0,4.0,NaN,4.0,NaN,0.0,2.0,4.0,4.0,9.0,27.0,8.0,14.0,4.0,13.0,0.0,0.0,0.0,6.0,95.0,0.0,1.0,0.0,372872.0,99468.0,15000.0,94072.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2018-12-01
2,5000,5000,5000.0,36 months,17.97,180.69,D,D1,Administrative,6 years,MORTGAGE,59280.0,Source Verified,Dec-2018,Current,n,NaN,debt_consolidation,Debt consolidation,490xx,

---
## Milestone 3: Cleaning the Website Data

**Data Source:** NYU Stern — Damodaran Default Spreads by Credit Rating  
**Retrieved From:** [pages.stern.nyu.edu](https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ratings.html)  
**Description:** A table of interest coverage ratios mapped to credit ratings and default
spreads, maintained by Professor Aswath Damodaran. This data serves as a bridge between
LendingClub's internal letter grades and standardized S&P credit ratings with associated
default risk premiums.  
**Relationship to Project:** Joins to the LendingClub data via the `grade` / `major_grade` column.

In [36]:
# =============================================================================
# Milestone 3 — Load Website Data
# Scraping the ratings table directly from NYU Stern on each run so the data
# reflects the most current published spreads.
# =============================================================================

url = 'https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ratings.html'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

tables = pd.read_html(response.text)
df_web = tables[0]

print("Original Column Names as Parsed from HTML:")
print(df_web.columns.tolist())
print(f"\nRaw Shape: {df_web.shape}")
df_web.head()

Original Column Names as Parsed from HTML:
[0, 1, 2, 3, 4, 5, 6, 7, 8]

Raw Shape: (19, 9)


/var/folders/c8/jzb_kqq53379zk0mw9gz8rvr0000gn/T/ipykernel_9198/1692769933.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


,0,1,2,3,4,5,6,7,8
0,If you want to update the spreads listed belo...,If you want to update the spreads listed belo...,If you want to update the spreads listed belo...,If you want to update the spreads listed belo...,If you want to update the spreads listed belo...,NaN,NaN,NaN,NaN
1,For large non-financial service firms,For large non-financial service firms,For large non-financial service firms,NaN,NaN,For financial service firms (default spreads ...,For financial service firms (default spreads ...,For financial service firms (default spreads ...,For financial service firms (default spreads ...
2,If interest coverage ratio is,NaN,NaN,NaN,NaN,If long term interest coverage ratio is,If long term interest coverage ratio is,If long term interest coverage ratio is,NaN
3,>,≤ to,Rating is,Spread is,NaN,greater than,≤ to,Rating is,Spread is
4,-100000,0.199999,D2/D,19.00%,NaN,-100000,0.049999,D2/D,19.00%


In [37]:
# =============================================================================
# Milestone 3 — Transformation Steps
# =============================================================================

# Step #1: Positional Column Selection
# The raw HTML scrape returns 9 columns from two side-by-side tables with no
# consistent headers. Selecting only the first 4 columns isolates the
# non-financial firm data which is relevant to this project.
df_web = df_web.iloc[:, :4].copy()

# Step #2: Schema Standardization — Rename Headers
# Replacing numeric column indices (0, 1, 2, 3) with descriptive names
# to make the data programmatically accessible and self-documenting.
df_web.columns = ['min_ratio', 'max_ratio', 'rating', 'spread']

# Step #3: Data Type Coercion
# The spread column contains percentage strings (e.g., '9.00%'). Stripping
# the '%' character and converting to numeric allows mathematical operations.
# errors='coerce' safely converts any non-numeric rows (headers, footers) to NaN.
df_web['spread_text'] = df_web['spread'].str.replace('%', '', regex=False)
df_web['spread_numeric'] = pd.to_numeric(df_web['spread_text'], errors='coerce')

# Step #4: Filter Non-Data Rows and Reset Index
# Rows that could not be converted to numeric (descriptive text, footers)
# became NaN during Step #3. Dropping them purges all non-data content.
df_web = df_web.dropna(subset=['spread_numeric']).reset_index(drop=True)

# Step #5: Create the Grade Bridge Column
# The rating column contains combined Moody's/S&P values (e.g., 'Aaa/AAA').
# Splitting on '/' and extracting the first character of the S&P side
# produces a single letter (A, B, C, etc.) that aligns with LendingClub's
# grade column — this is the relational key for the Milestone 5 join.
df_web['sp_rating'] = df_web['rating'].str.split('/').str[1]
df_web['major_grade'] = df_web['sp_rating'].str.strip().str[0].str.upper()

# Step #6: Numerical Normalization
# Converting whole-number spread values to decimal format (e.g., 9.00 → 0.09)
# to ensure accuracy in any downstream risk calculations.
df_web['spread'] = df_web['spread_numeric'] / 100
df_ratings = df_web[['min_ratio', 'max_ratio', 'rating', 'spread', 'major_grade']].copy()

print(f"Final Dataset Shape: {df_ratings.shape}")
print("\nFirst 5 Rows of Cleaned Ratings Data:")
df_ratings.head()

Final Dataset Shape: (15, 5)

First 5 Rows of Cleaned Ratings Data:


,min_ratio,max_ratio,rating,spread,major_grade
0,-100000,0.199999,D2/D,0.1900,D
1,0.2,0.649999,C2/C,0.1600,C
2,0.65,0.799999,Ca2/CC,0.1261,C
3,0.8,1.249999,Caa/CCC,0.0885,C
4,1.25,1.499999,B3/B-,0.0509,B


---
## Milestone 4: Connecting to the FRED API

**Data Source:** Federal Reserve Economic Data (FRED) — U.S. Civilian Unemployment Rate  
**Series ID:** `UNRATE` (Monthly, Seasonally Adjusted)  
**Retrieved From:** [Federal Reserve Bank of St. Louis](https://fred.stlouisfed.org/series/UNRATE)  
**API Documentation:** [fred.stlouisfed.org/docs/api](https://fred.stlouisfed.org/docs/api/api_key.html)  
**Description:** Monthly national unemployment rate published by the Bureau of Labor Statistics
and distributed via the FRED API. Date range is restricted to 2007–2018 to align with the
LendingClub loan issuance dates.  
**Relationship to Project:** Joins to the LendingClub data via `issue_d` / `year_month`.  
**Note:** A free API key is required. Replace `YOUR_FRED_API_KEY` below with your own key.

In [39]:
# =============================================================================
# Milestone 4 — Connect to FRED API and Load Raw Data
# Pulling the UNRATE series for the full 2007–2018 range to align with
# the updated LendingClub loan issuance window.
# =============================================================================

API_KEY  = "10216eff789a23d63506daa85727ab6d"
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

params = {
    "series_id"         : "UNRATE",
    "api_key"           : API_KEY,
    "file_type"         : "json",
    "observation_start" : "2007-01-01",
    "observation_end"   : "2018-12-01",
    "frequency"         : "m"
}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()
raw_json = response.json()

df_fred = pd.DataFrame(raw_json['observations'])

print(f"Status Code          : {response.status_code}")
print(f"Total Observations   : {raw_json['count']}")
print(f"\nRaw Column Names     : {df_fred.columns.tolist()}")
print(f"Raw Data Types:\n{df_fred.dtypes}")
print("\nFirst 3 Raw Observations:")
print(df_fred.head(3))

Status Code          : 200
Total Observations   : 144

Raw Column Names     : ['realtime_start', 'realtime_end', 'date', 'value']
Raw Data Types:
realtime_start    object
realtime_end      object
date              object
value             object
dtype: object

First 3 Raw Observations:
  realtime_start realtime_end        date value
0     2026-03-05   2026-03-05  2007-01-01   4.6
1     2026-03-05   2026-03-05  2007-02-01   4.5
2     2026-03-05   2026-03-05  2007-03-01   4.4


In [40]:
# =============================================================================
# Milestone 4 — Transformation Steps
# =============================================================================

# Step #1: Replace Column Headers
# FRED returns generic labels ('date', 'value'). Renaming them to descriptive
# names that clearly reflect their role in this project.
df_fred = df_fred.rename(columns={
    "date"  : "observation_date",
    "value" : "unemployment_rate_raw"
})

# Step #2: Drop Internal FRED Timestamp Columns
# 'realtime_start' and 'realtime_end' are FRED's internal data versioning
# timestamps and carry no analytical value for this project.
df_fred = df_fred.drop(columns=["realtime_start", "realtime_end"])

# Step #3: Fix Data Types
# Both columns arrived as strings. Converting 'observation_date' to datetime
# enables date-based operations. Converting 'unemployment_rate_raw' to float
# enables mathematical analysis. FRED occasionally returns '.' for missing
# values — errors='coerce' handles those safely by converting them to NaN.
df_fred["observation_date"]      = pd.to_datetime(df_fred["observation_date"])
df_fred["unemployment_rate_raw"] = pd.to_numeric(
    df_fred["unemployment_rate_raw"], errors="coerce"
)

# Step #4: Identify Outliers and Bad Data
# The U.S. unemployment rate has historically remained between 3% and 15%.
# Any values outside this range are flagged for inspection rather than
# silently removed, in order to preserve a full audit trail.
lower_bound = 3.0
upper_bound = 15.0

outliers = df_fred[
    (df_fred["unemployment_rate_raw"] < lower_bound) |
    (df_fred["unemployment_rate_raw"] > upper_bound) |
    (df_fred["unemployment_rate_raw"].isna())
]

print(f"Step #4 — Outliers or Missing Values Flagged: {len(outliers)}")
if len(outliers) == 0:
    print("No outliers detected. All values fall within the expected historical range.")

print(f"\nDescriptive Statistics:")
print(df_fred["unemployment_rate_raw"].describe())

# Step #5: Remove Duplicate Records
# Each row should represent exactly one calendar month. Duplicate dates would
# indicate a malformed API response. Keeping the first occurrence.
before = len(df_fred)
df_fred = df_fred.drop_duplicates(subset=["observation_date"], keep="first")
print(f"\nStep #5 — Duplicates Removed: {before - len(df_fred)}")

# Step #6: Format Data for Readability and Join Compatibility
# Creating 'year_month' in YYYY-MM format to serve as the join key to the
# LendingClub 'issue_d' column. Rounding the rate to one decimal place and
# adding a human-readable category label for downstream analysis.
df_fred["year_month"]            = df_fred["observation_date"].dt.strftime("%Y-%m")
df_fred["unemployment_rate_pct"] = df_fred["unemployment_rate_raw"].round(1)

def categorize_rate(rate):
    if pd.isna(rate)  : return "Unknown"
    elif rate < 5.0   : return "Low"
    elif rate < 7.0   : return "Moderate"
    elif rate < 10.0  : return "High"
    else              : return "Crisis"

df_fred["rate_category"] = df_fred["unemployment_rate_pct"].apply(categorize_rate)
df_fred = df_fred.drop(columns=["unemployment_rate_raw"])

df_unemployment = df_fred[
    ["observation_date", "year_month", "unemployment_rate_pct", "rate_category"]
].reset_index(drop=True)

print(f"\nRate Category Distribution:")
print(df_unemployment["rate_category"].value_counts())
print(f"\nFinal Dataset Shape: {df_unemployment.shape}")
print("\nFirst 5 Rows of Cleaned Unemployment Data:")
df_unemployment.head()

Step #4 — Outliers or Missing Values Flagged: 0
No outliers detected. All values fall within the expected historical range.

Descriptive Statistics:
count    144.000000
mean       6.519444
std        2.001268
min        3.700000
25%        4.700000
50%        6.100000
75%        8.300000
max       10.000000
Name: unemployment_rate_raw, dtype: float64

Step #5 — Duplicates Removed: 0

Rate Category Distribution:
rate_category
High        58
Low         45
Moderate    40
Crisis       1
Name: count, dtype: int64

Final Dataset Shape: (144, 4)

First 5 Rows of Cleaned Unemployment Data:


,observation_date,year_month,unemployment_rate_pct,rate_category
0,2007-01-01,2007-01,4.6,Low
1,2007-02-01,2007-02,4.5,Low
2,2007-03-01,2007-03,4.4,Low
3,2007-04-01,2007-04,4.5,Low
4,2007-05-01,2007-05,4.4,Low


---
## Milestone 5: Merging the Data and Storing in a Database

**Objective:** Load all three cleaned datasets into a SQLite relational database as individual
tables, join them into a single unified dataset using SQL, and export the result for
visualization in Tableau.

**Join Logic:**
- `loans` → `ratings` via `grade` / `major_grade` (LendingClub letter grade maps to S&P rating)
- `loans` → `unemployment` via `year_month` (loan issue date maps to unemployment month)

**Database:** `lending_project.db` (SQLite, created locally on first run)  
**Export:** `lending_merged_final.csv` (used as the Tableau data source)

In [41]:
# =============================================================================
# Milestone 5 — Step 1: Prepare Join Key on Loans Table
# The 'issue_d' column is formatted as 'Dec-2018' in this dataset version.
# Converting to 'YYYY-MM' format to align with the FRED unemployment
# 'year_month' column for the SQL join.
# =============================================================================

df_loans['year_month'] = df_loans['issue_d_parsed'].dt.strftime('%Y-%m')

print("Sample 'issue_d' to 'year_month' Conversion:")
print(df_loans[['issue_d', 'year_month']].drop_duplicates().sort_values('year_month').head(10))
print(f"\nDate Range : {df_loans['year_month'].min()} to {df_loans['year_month'].max()}")

Sample 'issue_d' to 'year_month' Conversion:
          issue_d year_month
2118220  Jun-2007    2007-06
2118185  Jul-2007    2007-07
2118156  Aug-2007    2007-08
2118139  Sep-2007    2007-09
2118093  Oct-2007    2007-10
2118046  Nov-2007    2007-11
2117971  Dec-2007    2007-12
2117767  Jan-2008    2008-01
2117613  Feb-2008    2008-02
2117350  Mar-2008    2008-03

Date Range : 2007-06 to 2018-12


In [42]:
# =============================================================================
# Milestone 5 — Step 2: Load Cleaned Datasets into SQLite
# Writing each cleaned DataFrame as an individual table in the database.
# if_exists='replace' ensures the notebook can be re-run cleanly without
# conflicts from prior executions.
# =============================================================================

conn = sqlite3.connect('lending_project.db')

df_loans.to_sql('loans',        conn, if_exists='replace', index=False)
df_ratings.to_sql('ratings',    conn, if_exists='replace', index=False)
df_unemployment.to_sql('unemployment', conn, if_exists='replace', index=False)

# Verify all three tables were created successfully
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("Tables Successfully Loaded into lending_project.db:")
for table in tables:
    print(f"  - {table[0]}")

# Print row counts for each table as a sanity check
for table_name in ['loans', 'ratings', 'unemployment']:
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"  {table_name}: {count:,} rows")

Tables Successfully Loaded into lending_project.db:
  - merged_final
  - loans
  - ratings
  - unemployment
  loans: 2,238,121 rows
  ratings: 15 rows
  unemployment: 144 rows


In [45]:
# =============================================================================
# Milestone 5 — Step 3: Merge All Three Tables Using SQL Joins
# Joining all three tables into a single unified dataset.
#
# LEFT JOIN is used in both cases so that loan records are preserved even if
# a grade or date does not have a matching entry in the ratings or unemployment
# tables — this prevents unintended data loss.
#
# The ratings table is aggregated by major_grade first (averaging spread per
# grade) because multiple sub-ratings map to the same letter grade, which
# would otherwise produce duplicate loan rows after the join.
# =============================================================================

query = """
    SELECT
        l.loan_amnt,
        l.int_rate,
        l.grade,
        l.issue_d,
        l.year_month,
        l.annual_inc,
        l.loan_status,
        l.purpose,
        l.addr_state,
        l.emp_title,
        l.dti,
        l.term,
        r.rating,
        r.avg_spread         AS spread,
        u.unemployment_rate_pct,
        u.rate_category
    FROM loans l
    LEFT JOIN (
        SELECT
            major_grade,
            AVG(spread)  AS avg_spread,
            MAX(rating)  AS rating
        FROM ratings
        GROUP BY major_grade
    ) r ON l.grade = r.major_grade
    LEFT JOIN unemployment u
        ON l.year_month = u.year_month
"""

df_merged = pd.read_sql_query(query, conn)

print(f"Merged Dataset Shape : {df_merged.shape}")
print(f"\nNull Values per Column:")
print(df_merged.isnull().sum())
print(f"\nFirst 5 Rows of Merged Dataset:")
df_merged.head()

Merged Dataset Shape : (2238121, 16)

Null Values per Column:
loan_amnt                     0
int_rate                      0
grade                         0
issue_d                       0
year_month                    0
annual_inc                    0
loan_status                   0
purpose                       0
addr_state                    0
emp_title                     0
dti                        1711
term                          0
rating                   188301
spread                   188301
unemployment_rate_pct         0
rate_category                 0
dtype: int64

First 5 Rows of Merged Dataset:


,loan_amnt,int_rate,grade,issue_d,year_month,annual_inc,loan_status,purpose,addr_state,emp_title,dti,term,rating,spread,unemployment_rate_pct,rate_category
0,2500,13.56,C,Dec-2018,2018-12,55000.0,Current,debt_consolidation,NY,Chef,18.24,36 months,Caa/CCC,0.124867,3.9,Low
1,30000,18.94,D,Dec-2018,2018-12,90000.0,Current,debt_consolidation,LA,Postmaster,26.52,60 months,D2/D,0.190000,3.9,Low
2,5000,17.97,D,Dec-2018,2018-12,59280.0,Current,debt_consolidation,MI,Administrative,10.51,36 months,D2/D,0.190000,3.9,Low
3,4000,18.94,D,Dec-2018,2018-12,92000.0,Current,debt_consolidation,WA,IT Supervisor,16.74,36 months,D2/D,0.190000,3.9,Low
4,30000,16.14,C,Dec-2018,2018-12,57250.0,Current,debt_consolidation,MD,Mechanic,26.35,60 months,Caa/CCC,0.124867,3.9,Low


In [46]:
# =============================================================================
# Milestone 5 — Step 4: Save Merged Dataset to Database and Export for Tableau
# Writing the merged dataset back to the database as its own table for
# future reference, then exporting to CSV for use in Tableau.
# =============================================================================

# Save as a table in the database
df_merged.to_sql('merged_final', conn, if_exists='replace', index=False)

# Export to CSV for Tableau
df_merged.to_csv('lending_merged_final.csv', index=False)

# Confirm the export
print("lending_merged_final.csv exported successfully.")
print(f"\nDatabase Tables:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
for table in cursor.fetchall():
    print(f"  - {table[0]}")

conn.close()
print("\nDatabase connection closed.")

lending_merged_final.csv exported successfully.

Database Tables:
  - loans
  - ratings
  - unemployment
  - merged_final

Database connection closed.


In [47]:
# =============================================================================
# Milestone 5 — Final Output: Human-Readable Merged Dataset
# =============================================================================

pd.set_option('display.max_columns', None)

print("=" * 60)
print("   FINAL MERGED DATASET — LENDING PROJECT")
print("=" * 60)
print(f"  Total Records         : {len(df_merged):,}")
print(f"  Total Columns         : {df_merged.shape[1]}")
print(f"  Loan Date Range       : {df_merged['issue_d'].min()} to {df_merged['issue_d'].max()}")
print(f"  Loan Grade Values     : {sorted(df_merged['grade'].dropna().unique())}")
print(f"  Unemployment Range    : {df_merged['unemployment_rate_pct'].min()}% — {df_merged['unemployment_rate_pct'].max()}%")
print(f"  Unique Loan Purposes  : {df_merged['purpose'].nunique()}")
print(f"  Unique States         : {df_merged['addr_state'].nunique()}")
print("=" * 60)

df_merged.head(20)

   FINAL MERGED DATASET — LENDING PROJECT
  Total Records         : 2,238,121
  Total Columns         : 16
  Loan Date Range       : Apr-2008 to Sep-2018
  Loan Grade Values     : ['A', 'B', 'C', 'D', 'E', 'F', 'G']
  Unemployment Range    : 3.7% — 10.0%
  Unique Loan Purposes  : 14
  Unique States         : 51


,loan_amnt,int_rate,grade,issue_d,year_month,annual_inc,loan_status,purpose,addr_state,emp_title,dti,term,rating,spread,unemployment_rate_pct,rate_category
0,2500,13.56,C,Dec-2018,2018-12,55000.0,Current,debt_consolidation,NY,Chef,18.24,36 months,Caa/CCC,0.124867,3.9,Low
1,30000,18.94,D,Dec-2018,2018-12,90000.0,Current,debt_consolidation,LA,Postmaster,26.52,60 months,D2/D,0.190000,3.9,Low
2,5000,17.97,D,Dec-2018,2018-12,59280.0,Current,debt_consolidation,MI,Administrative,10.51,36 months,D2/D,0.190000,3.9,Low
3,4000,18.94,D,Dec-2018,2018-12,92000.0,Current,debt_consolidation,WA,IT Supervisor,16.74,36 months,D2/D,0.190000,3.9,Low
4,30000,16.14,C,Dec-2018,2018-12,57250.0,Current,debt_consolidation,MD,Mechanic,26.35,60 months,Caa/CCC,0.124867,3.9,Low
5,5550,15.02,C,Dec-2018,2018-12,152500.0,Current,credit_card,IN,Director COE,37.94,36 months,Caa/CCC,0.124867,3.9,Low
6,2000,17.97,D,Dec-2018,2018-12,51000.0,Current,debt_consolidation,IL,Account Manager,2.40,36 months,D2/D,0.190000,3.9,Low
7,6000,13.56,C,Dec-2018,2018-12,65000.0,Current,credit_card,IN,Assistant Director,30.10,36 months,Caa/CCC,0.124867,3.9,Low
8,5000,17.97,D,Dec-2018,2018-12,53580.0,Current,debt_consolidation,FL,Legal Assistant III,21.16,36 months,D2/D,0.190000,3.9,Low
9,5500,22.35,D,Dec-2018,2018-12,50000.0,Current,credit_card,LA,Unknown,15.94,36 months,D2/D,0.190000,3.9,Low


---
### Tableau Visualizations

| # | Title | Data Sources | Link |
|---|-------|-------------|------|
| 1 | Average Interest Rate by Loan Grade | Loans | [View on Tableau Public](https://public.tableau.com/app/profile/kat.chu/viz/AvgInterestRatebyGrade/AvgInterestRatebyGrade) |
| 2 | Loan Volume Over Time | Loans | [View on Tableau Public](https://public.tableau.com/app/profile/kat.chu/viz/LoanVolumeOverTime/LoanVolumeOverTime) |
| 3 | Loan Volume vs Unemployment Rate | Loans + Unemployment | [View on Tableau Public](https://public.tableau.com/app/profile/kat.chu/viz/LoanVolumevsUnemploymentRate/LoanVolumevsUnemploymentRate) |
| 4 | Average Loan Amount and Interest Rate by Grade and Purpose | Loans + Ratings | [View on Tableau Public](https://public.tableau.com/app/profile/kat.chu/viz/AvgLoanAmountandInterestRatebyGradeandPurpose/LoanPortfolioRisk) |
| 5 | Interest Rate Pricing by Economic Condition | Loans + Unemployment | [View on Tableau Public](https://public.tableau.com/app/profile/kat.chu/viz/InterestRatePricingbyEconomicCondition/InterestRatePricingbyEconomicCondition) |

A PDF of all five visualizations is included with this submission as `mile5_visualizations_KChu.pdf`.

**What did the visualizations show?**

Interest rates rise steadily from grade A (~7%) to grade G (~28%), confirming LendingClub's 
grading system is strongly tied to pricing. Loan volume grew exponentially from 2007 to 2018 
with notable volatility after 2015. The dual-axis chart comparing loan volume against 
unemployment showed a compelling inverse relationship — as unemployment peaked during the 
2008 financial crisis loan volume was nearly flat, then exploded as the economy recovered 
through 2018. The default spread heat map confirmed grade D loans carry nearly 19% default 
risk premium versus less than 1% for grade A. The interest rate pricing by economic condition 
chart showed that even grade A borrowers paid higher rates during crisis-level unemployment, 
suggesting macroeconomic conditions influence pricing across all risk tiers.

### Project Summary

**What did you do to complete the project?**

For this project I pulled together three data sources: a LendingClub loan CSV from Kaggle, 
credit rating spreads scraped from NYU Stern, and monthly U.S. unemployment rates from the 
FRED API; cleaned each one individually, loaded them into a SQLite database as separate 
tables, and joined them into a single unified dataset for analysis in Tableau.

**What did you learn?**

The biggest takeaway from this project is that working with multiple data sources is 
significantly harder than working with one. Every source arrived in a completely different 
format, with different conventions, different date formats, different column naming, and 
different levels of cleanliness; and none of them were designed to work together. Making 
them compatible required a lot of attention to detail and careful decision making at every 
step. A single mismatched date format or an unconverted data type would silently break a 
join without throwing an obvious error, which meant constantly verifying outputs and 
cross-checking results.

I also learned that data cleaning is not a linear process, decisions made early, like how 
to handle missing employment titles or how to normalize the date format, have downstream 
consequences that only become visible much later when trying to merge or visualize. The NYU 
Stern scrape was a good example of this: the raw HTML table had no usable headers, numeric 
data stored as percentage strings, and two completely different rating systems merged into 
one column. Extracting a single usable join key from that required understanding both the 
structure of the HTML and the logic of the rating system itself. Similarly, aligning 
LendingClub's 'Dec-2018' date format with FRED's 'YYYY-MM-DD' format required building an 
intermediate key column just to make the SQL join possible, the kind of small but critical 
detail that takes real troubleshooting to figure out. This project made clear that clean, 
merged, analysis-ready data is never a given; it is always the result of deliberate, 
methodical work.

**What changes were made to the data?**

Across all three sources, significant cleaning and transformation work was required. For the 
LendingClub CSV, empty columns were dropped, duplicate records were removed, column headers 
were stripped of whitespace, missing employment titles were filled with 'Unknown', and the 
top 1% of income outliers were removed to prevent skewing. The issue_d date column was 
converted from 'Dec-2018' string format to a parsed datetime and then reformatted to YYYY-MM 
to serve as a join key. 

For the NYU Stern data, the raw HTML scrape returned 9 columns with 
no consistent headers, only the first 4 were selected by position, columns were renamed 
descriptively, percentage strings were stripped and converted to numeric floats, non-data 
rows like headers and footers were removed using NaN filtering, and a major_grade column was 
engineered by splitting the combined Moody's/S&P rating strings to extract a single letter 
grade as the join key to the loans table. Spread values were also converted from whole 
numbers to decimals for mathematical accuracy. 

For the FRED API data, all columns arrived 
as strings; the date column was converted to datetime, the unemployment rate was converted 
to float, internal FRED timestamp columns were dropped, duplicates were checked for, the 
date was reformatted to YYYY-MM as the join key, values were rounded to one decimal place, 
and a rate_category label was added to make the data more human-readable for analysis.

**Are there any legal or regulatory guidelines for your data or project topic?**

Yes, this dataset falls under the Fair Credit Reporting Act (FCRA) and Equal Credit 
Opportunity Act (ECOA) since variables like loan grade, zip code, and purpose can proxy 
for protected characteristics such as race and gender.

**What risks could be created based on the transformations done?**

The rate_category labels introduce boundary bias since a loan at 6.9% and 7.0% unemployment 
would be treated as categorically different despite nearly identical conditions. Using a 
national unemployment rate also smooths over real geographic variation in economic conditions.

**Did you make any assumptions in cleaning/transforming the data?**

Several assumptions were made throughout the project. When employment titles were missing, 
I assumed those borrowers were not meaningfully different from borrowers who provided one, 
so filling with 'Unknown' was safe rather than dropping those rows entirely. I assumed a 
national unemployment rate was a close enough representation of economic conditions even 
though borrowers across 51 states experienced very different local economies. I also assumed 
that grouping all S&P sub-ratings under a single letter grade; for example treating AA+, 
AA, and AA- all as 'A', was a reasonable simplification for joining the two datasets, even 
though it loses some granularity in the credit risk data. Finally I assumed that removing 
the top 1% of incomes correctly identified outliers rather than accidentally removing a 
legitimate segment of high-earning borrowers.

**How was your data sourced/verified for credibility?**

LendingClub data was sourced from a verified Kaggle dataset. NYU Stern ratings are maintained 
by Professor Aswath Damodaran, a widely cited authority in financial valuation. FRED data is 
published by the Federal Reserve and sourced directly from the Bureau of Labor Statistics.

**Was your data acquired in an ethical way?**

Yes, all three sources are publicly available and intended for research and educational use. 
The LendingClub data is anonymized, NYU Stern publishes freely for academic use, and FRED 
is a public government API requiring only a free registration key.

**How would you mitigate any of the ethical implications you have identified?**

The raw unemployment rate is retained alongside category labels so future analysis is not 
locked into subjective thresholds. This dataset should not be used in any credit-scoring 
model without a full disparate impact audit first. All transformation decisions are 
documented here to support transparency and reproducibility.